# Statistical model - Mixed effects model

Fit data into a hierarchical model -- take advantage that nearby countries should show similar behaviors.

This should allow to reduce the error in predicteing for countries with small trinaing samples

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.special import logit
from sklearn.model_selection import GroupShuffleSplit



In [2]:
train_df_all = pd.read_csv("train_augmented.csv",index_col=0)
train_df_all.columns

Index(['category_name', 'geography_name', 'wave_date', 'question_text',
       'qualifying_options', 'disqualifying_options', 'weight_3',
       'base_weight_3', 'percentage_3', 'timeframe_months', 'have_you_quest',
       'are_you_quest', 'multi_choice_quest', 'do_you_own_quest',
       'n_qualifying', 'n_disqualifying', 'qualifying_has_no',
       'qualifying_has_yes', 'continent', 'product_category', 'broad_category',
       'is_luxury', 'is_subscription', 'audience_breadth',
       'purchase_frequency', 'usage_frequency', 'cost_tier'],
      dtype='str')

In [3]:
train_df_all["logit_incidence"] = logit( train_df_all["percentage_3"] )
train_df_all["logit_incidence"].describe()

count    829.000000
mean       0.298250
std        0.995341
min       -3.009467
25%       -0.447312
50%        0.159980
75%        0.942979
max        6.868457
Name: logit_incidence, dtype: float64

Ok, no infifinite values!



In [4]:
def mixed_effect_nested_cols(df):
    """
    Add nesting columns to country and product features
    """
    # random effects in single group
    df["all"] = 1
    
    # add nested features
    df["continent_country"] = (
        df["continent"].astype(str) + ":" +
        df["geography_name"].astype(str)
    )
    
    df["product_group"] = (
        df["broad_category"].astype(str) + ":" +
        df["product_category"].astype(str)
    )
    return df

train_df_all = mixed_effect_nested_cols(train_df_all)

In [5]:
train_df_all.logit_incidence.describe()

count    829.000000
mean       0.298250
std        0.995341
min       -3.009467
25%       -0.447312
50%        0.159980
75%        0.942979
max        6.868457
Name: logit_incidence, dtype: float64

## Split dataset into train and validation sets

We want the validation split to have completely new product categories, as will be the case for the test set

In [6]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, val_idx = next(splitter.split(
    train_df_all,
    groups=train_df_all["product_category"]
))

train = train_df_all.iloc[train_idx].copy()
val = train_df_all.iloc[val_idx].copy()

print(f"Train dataset: {len(train)} rows")
print(f"Validation dataset: {len(val)} rows")

Train dataset: 653 rows
Validation dataset: 176 rows


Now define the variance structure of the model with nesting

In [7]:
vc_formula = {
    "continent": "0 + C(continent)",
    "country": "0 + C(continent_country)",
    "broad_category": "0 + C(broad_category)",
    "product": "0 + C(product_category)",
}


In [8]:
model = smf.mixedlm(
    "logit_incidence ~ cost_tier + audience_breadth + n_qualifying + n_disqualifying",
    data=train,
    groups="all",
    re_formula="0",
    vc_formula=vc_formula,
)

In [9]:
result = model.fit(method="lbfgs")
print(result.summary())

            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: logit_incidence
No. Observations: 653     Method:             REML           
No. Groups:       1       Scale:              0.0894         
Min. group size:  653     Log-Likelihood:     -556.6321      
Max. group size:  653     Converged:          Yes            
Mean group size:  653.0                                      
-------------------------------------------------------------
                   Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------
Intercept          -1.679    0.262 -6.409 0.000 -2.193 -1.166
cost_tier          -0.090    0.035 -2.560 0.010 -0.158 -0.021
audience_breadth    0.610    0.040 15.166 0.000  0.531  0.688
n_qualifying        0.032    0.023  1.401 0.161 -0.013  0.076
n_disqualifying    -0.092    0.037 -2.499 0.012 -0.164 -0.020
broad_category Var  0.054    0.091                           
continent Var       

/tmp/ipykernel_127443/2921328197.py:1: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  result = model.fit(method="lbfgs")


Save random effect coefficients for inspection

In [28]:
result.random_effects[1].to_frame().to_csv("lmer_random_eff.csv")

## Use the mixed-effect model for prediction

This is slightly more convoluted than R's lmer, I had some help from ChatGPT...

In [29]:
def predict_mixed(result, newdata, effects):
    """
    effects: dictionary mapping variance-component names
             to their corresponding columns in newdata.
    """
    pred = pd.Series(
        result.predict(newdata),
        index=newdata.index,
        dtype=float,
    )

    # get the random effect coefficients
    re = result.random_effects[1]

    for effect_name, column in effects.items():
        prefix = f"{effect_name}[C({column})["

        # Extract level -> fitted random effect for nested effects
        mapping = {
            key[len(prefix):-2]: value
            for key, value in re.items()
            if key.startswith(prefix)
        }

        # predict (add all nested coefficients)
        pred += newdata[column].astype(str).map(mapping).fillna(0)

    return pred

In [13]:
pred_logit = predict_mixed(
    result,
    val,
    effects={
        "continent": "continent",
        "country": "continent_country",
        # Product is deliberately excluded:
        # evaluation contains unseen products.
    },
)

pred_incidence = 1 / (1 + np.exp(-pred_logit))

In [14]:
(pred_logit - val.logit_incidence).describe()

count    176.000000
mean      -0.146252
std        0.733960
min       -2.404705
25%       -0.588224
50%       -0.110734
75%        0.359409
max        1.695225
dtype: float64

In [15]:
val = val.copy()
val["error"] = np.asarray(pred_logit) - val["logit_incidence"]

product_errors = val.groupby("product_category")["error"].mean()

print(product_errors.describe())

count    104.000000
mean      -0.173518
std        0.757775
min       -2.404705
25%       -0.596792
50%       -0.117832
75%        0.370558
max        1.397565
Name: error, dtype: float64


## Estimate residuals on validation set

In [16]:
from scipy.stats import norm
import numpy as np

errors = pred_logit - val.logit_incidence


# for a new product, the expected variance of the prediction includes:
# - general variance of the prediction
residual_var = result.scale
# - variance of product means within a product group
product_var = [v for v, feat in zip(result.vcomp, result.model.exog_vc.names) if feat == "product"][0]
sigma_pred = np.sqrt(product_var + residual_var)
margin = norm.ppf(0.90) * sigma_pred

print("80% CI for logit prediction:", margin)
mae = errors.abs().mean()
print("Mean Absolute Error:", mae)


80% CI for logit prediction: 0.8631990166819297
Mean Absolute Error: 0.582992624098943


We could take into account different variances for countries with less data but the the variance component for countries with continents is rather small (0.003)

## 80% coverage intervals

For validation intervals

In [17]:
from scipy.special import expit

lower_80 = expit(pred_logit - margin)
upper_80 = expit(pred_logit + margin)

In [18]:
pd.DataFrame({
    "lower_80": lower_80,
    "point_estimate":pred_incidence,
    "upper_80":upper_80})

,lower_80,point_estimate,upper_80
0,0.147756,0.291294,0.493523
1,0.137342,0.274016,0.472242
3,0.355179,0.566319,0.755848
8,0.214943,0.393604,0.606115
13,0.375935,0.588159,0.771986
...,...,...,...
824,0.540076,0.735722,0.868418
825,0.518845,0.718820,0.858370
826,0.518845,0.718820,0.858370
827,0.633200,0.803635,0.906563


## Check that actual data is within coverage interval

In [19]:
coverage = (errors.abs() <= margin).mean()
print("80% interval coverage:", coverage)



80% interval coverage: 0.7613636363636364


# Apply to test dataset

In [20]:
test = pd.read_csv("test_augmented.csv")
test = mixed_effect_nested_cols(test)

In [21]:
test_pred_logit = predict_mixed(
    result,
    test,
    effects={
        "continent": "continent",
        "country": "continent_country",
        # Product is deliberately excluded:
        # evaluation contains unseen products.
    },
)

pred_incidence = 1 / (1 + np.exp(-test_pred_logit))
lower_80 = expit(test_pred_logit - margin)
upper_80 = expit(test_pred_logit + margin)

test_results = pd.DataFrame({
    "prediction_id": test.prediction_id,
    "point_estimate":pred_incidence,
    "lower_80": lower_80,
    "upper_80":upper_80}
)

In [22]:
test_results.to_csv("predictions.csv")

In [23]:
(test_results["lower_80"] < test_results["point_estimate"]).all()

np.True_

In [24]:
(test_results["upper_80"] > test_results["point_estimate"]).all()

np.True_

In [25]:
train_df_all[train_df_all["category_name"].str.contains("Condition")]

,category_name,geography_name,wave_date,question_text,qualifying_options,disqualifying_options,weight_3,base_weight_3,percentage_3,timeframe_months,...,is_luxury,is_subscription,audience_breadth,purchase_frequency,usage_frequency,cost_tier,logit_incidence,all,continent_country,product_group
8,Air Conditioners,Australia,2024-09-01,In the past 2 years have you purchased an air ...,"No, but I am open or likely to in the near fut...",No,681.01,1966.48,0.34631,24.0,...,False,False,3,every_few_years,daily,4,-0.635299,1,Asia-Pacific:Australia,Home & Furnishings:Air Conditioners


In [26]:
prods_in_both_conutries = train_df_all.groupby("product_category").apply(
    lambda grp: (grp.geography_name == "Australia").any() &  (grp.geography_name == "United States").any()
)

In [27]:
prods_in_both_conutries[prods_in_both_conutries]

product_category
Air Fresheners & Home Fragrances    True
Alcohol Free Beer                   True
Asian Foods & Condiments            True
Audio Content                       True
Baby and Child Products             True
                                    ... 
Underwear, socks, and basics        True
Virtual Assistants                  True
Vitamins and Dietary Supplements    True
Wine                                True
Yoghurt                             True
Length: 79, dtype: bool